# ALeRCE TAP — LSST transients

ALeRCE now serves its database through an IVOA **TAP** endpoint at `https://tap.alerce.online/tap`,
which is how you reach its **LSST** data: the REST client in `queryALeRCE.ipynb` only talks to the
legacy ZTF v1 API. TAP takes ADQL (SQL plus cone-search functions), needs no account or token, and
works from TOPCAT, pyvo, astroquery or plain HTTP.

Two schemas are published:

- **`alerce_tap`** — the multi-survey catalogue, which holds the LSST tables
- **`ztf`** — the legacy ZTF catalogue

Survey ids (`sid`) inside `alerce_tap`: **0 = ZTF, 1 = LSST DIA Object, 2 = LSST Solar System**.

The LSST tables are `lsst_dia_object` (83 columns of per-object summary photometry),
`lsst_detection` (per-alert measurements), `lsst_forced_photometry`, `lsst_ss_object` and
`lsst_mpc_orbits`, plus the survey-agnostic `detection`, `probability`, `taxonomy` and `xmatch`.

In [ ]:
import io

import pandas as pd
import requests
from astropy.time import Time

TAP_URL = "https://tap.alerce.online/tap"

### The query helper

`pyvo` is the idiomatic TAP client and works against this service, but it imports
`astropy.coordinates`, which is broken in this environment (a mixed astropy install that predates
this notebook). Plain HTTP is a two-line alternative and is what the helper below uses — a TAP
sync query is just a GET with an ADQL string.

```python
# once astropy is repaired, this is equivalent:
import pyvo
tap = pyvo.dal.TAPService(TAP_URL)
df = tap.search("SELECT TOP 10 oid, ra, dec FROM alerce_tap.lsst_dia_object").to_table().to_pandas()
```

In [ ]:
def tap_query(adql, maxrec=100_000, timeout=600, warn=True):
    """Run an ADQL query against ALeRCE TAP and return a DataFrame.

    The service silently truncates at `maxrec` rows, so a result of exactly that
    length is reported as suspect rather than treated as complete.
    """
    r = requests.get(
        f"{TAP_URL}/sync",
        params={"LANG": "ADQL", "REQUEST": "doQuery", "RESPONSEFORMAT": "csv",
                "MAXREC": maxrec, "QUERY": adql},
        timeout=timeout,
    )
    r.raise_for_status()
    df = pd.read_csv(io.BytesIO(r.content))
    if warn and len(df) == maxrec:
        print(f"Warning: got exactly MAXREC={maxrec} rows — the result is probably truncated.")
    return df


tap_query("SELECT TOP 3 oid, ra, dec, ndiasources FROM alerce_tap.lsst_dia_object", maxrec=10)

In [ ]:
# What is on offer. tap_schema is queryable like any other table.
tap_query("""
    SELECT table_name, description
    FROM tap_schema.tables
    WHERE schema_name = 'alerce_tap' AND table_name LIKE '%lsst%'
""", maxrec=100)

### Which table to drive off, and why

Two tables describe the same 5,435,408 LSST objects:

- **`alerce_tap.object`** (`sid = 1`) — id, mean position, `firstmjd`/`lastmjd`, detection counts.
  Fully populated: zero NULLs in `lastmjd` or `n_det`.
- **`alerce_tap.lsst_dia_object`** — the 83-column LSST summary, including per-band fluxes. Its
  `firstdiasourcemjdtai` and `lastdiasourcemjdtai` are **NULL for 4,989,529 of 5,435,408 rows
  (91.8%)**: every object whose last activity predates MJD 61235 (2026-08-13) is missing them.

So the time cut goes on `object.lastmjd` and `lsst_dia_object` is joined in for photometry. Cutting
on `lastdiasourcemjdtai` instead happens to give the same answer for recent windows — both return
140,150 objects for the last 7 days — but it silently drops everything older than MJD 61235, and
`ORDER BY lastdiasourcemjdtai DESC` sorts the NULLs to the top (PostgreSQL's default; ADQL here
rejects `NULLS LAST`).

MJD in the LSST tables is **TAI**, matching Lasair. ALeRCE's ZTF tables use UTC MJD — don't mix them.

In [ ]:
LSST_SID = 1
BANDS = {1: "g", 2: "r", 3: "i", 4: "z", 5: "y", 6: "u"}  # alerce_tap.band, sid = 1


def latest_detection():
    """(mjd, days_behind) of the most recent LSST detection in the ALeRCE database."""
    mjd = tap_query(
        "SELECT MAX(lastdiasourcemjdtai) AS mx FROM alerce_tap.lsst_dia_object", maxrec=10
    ).mx.iloc[0]
    return mjd, Time.now().tai.mjd - mjd


mjd, lag = latest_detection()
print(f"latest LSST detection: MJD {mjd:.4f} = {Time(mjd, format='mjd', scale='tai').iso} "
      f"({lag:.2f} days ago)")

### Classifications

LSST alerts are scored by ALeRCE's Rubin stamp classifier. Several versions live in the
`classifier` table and **the retired ones still hold rows**, so pin the classifier id rather than
taking whatever comes back: `classifier_id = 1` stopped updating at MJD 61235, while
`classifier_id = 3` (`stamp_classifier_rubin_beta_20260421`, v2.0.2) covers all 5.4M objects and is
current. `classifier_id = 10` is a separate binary SN off-nuclear classifier.

In [ ]:
tap_query("SELECT * FROM alerce_tap.classifier WHERE tid = 1", maxrec=100)

In [ ]:
LSST_CLASSIFIER_ID = 3

CLASSES = tap_query(
    f"SELECT class_id, class_name FROM alerce_tap.taxonomy WHERE classifier_id = {LSST_CLASSIFIER_ID}",
    maxrec=100,
).set_index("class_id").class_name.to_dict()
CLASSES

### The query

In [ ]:
TAP_MAX_ROWS = 1_000_000  # server hard limit per query; see the note below

BASE_COLUMNS = "o.oid, o.meanra, o.meandec, o.firstmjd, o.lastmjd, o.n_det, o.n_forced"
PHOTOMETRY_COLUMNS = ("d.g_psffluxmean, d.r_psffluxmean, d.i_psffluxmean, "
                      "d.z_psffluxmean, d.ndiasources")
CLASS_COLUMNS = "p.class_id, p.probability"


def build_query(where=(), order_by="o.oid", limit=TAP_MAX_ROWS,
                with_class=True, with_photometry=True):
    """Assemble an ADQL query over the LSST tables, driven off alerce_tap.object."""
    columns = [BASE_COLUMNS]
    tables = [f"alerce_tap.object AS o"]
    conditions = [f"o.sid = {LSST_SID}", *where]

    if with_photometry:
        columns.append(PHOTOMETRY_COLUMNS)
        tables.append("JOIN alerce_tap.lsst_dia_object AS d ON d.oid = o.oid")
    if with_class:
        columns.append(CLASS_COLUMNS)
        tables.append("JOIN alerce_tap.probability AS p ON p.oid = o.oid")
        conditions += [f"p.sid = {LSST_SID}", "p.ranking = 1",
                       f"p.classifier_id = {LSST_CLASSIFIER_ID}"]

    return f"""
        SELECT TOP {limit} {', '.join(columns)}
        FROM {' '.join(tables)}
        WHERE {' AND '.join(conditions)}
        ORDER BY {order_by}
    """


def decorate(df, with_class=True):
    """Add class names and an age column to a result frame."""
    if df.empty:
        return df
    if with_class and "class_id" in df:
        df["class_name"] = df.class_id.map(CLASSES)
    df["days_ago"] = (Time.now().tai.mjd - df.lastmjd).round(2)
    return df

#### Everything in a time window

Paged by keyset on `oid`, so a window larger than the per-query ceiling still comes back whole.

In [ ]:
def count_recent(n_days, extra_conditions=None):
    """Number of LSST objects with a detection in the past n_days. One query."""
    where = f"sid = {LSST_SID} AND lastmjd > {Time.now().tai.mjd - n_days:.6f}"
    if extra_conditions:
        where += f" AND ({extra_conditions})"
    return int(tap_query(
        f"SELECT COUNT(*) AS n FROM alerce_tap.object WHERE {where}", maxrec=10).n.iloc[0])


def recent_lsst_objects(n_days, with_class=True, with_photometry=True, extra_conditions=None,
                        page_size=TAP_MAX_ROWS, max_pages=25, verbose=True):
    """Every LSST object with a detection in the past `n_days`, as a DataFrame."""
    mjd_min = Time.now().tai.mjd - n_days

    frames, last_oid = [], None
    for page in range(max_pages):
        where = [f"o.lastmjd > {mjd_min:.6f}"]
        if last_oid is not None:
            where.append(f"o.oid > {last_oid}")
        if extra_conditions:
            where.append(f"({extra_conditions})")

        chunk = tap_query(build_query(where, order_by="o.oid", limit=page_size,
                                      with_class=with_class, with_photometry=with_photometry),
                          maxrec=page_size, warn=False)
        frames.append(chunk)
        if verbose:
            print(f"page {page + 1}: +{len(chunk)} rows (total {sum(len(f) for f in frames)})")
        if len(chunk) < page_size:
            break
        last_oid = int(chunk.oid.iloc[-1])
    else:
        print(f"Warning: stopped at max_pages={max_pages}; results are truncated.")

    df = pd.concat(frames, ignore_index=True)
    if df.empty:
        _, lag = latest_detection()
        print(f"Nothing in the last {n_days} days — the most recent LSST detection "
              f"in ALeRCE is {lag:.2f} days old.")
        return df
    return decorate(df, with_class).sort_values("days_ago").reset_index(drop=True)

#### Or just take the top N

`TOP n` with no time cut is the cheaper way to get a bulk sample: one request, no paging.
**1,000,000 rows is the server's hard ceiling** — ask for 2,000,000 and you get 1,000,001 back,
which is DaCHS signalling overflow by returning one row past the limit. In CSV that is the only
hint you get; the VOTable response carries `QUERY_STATUS=OVERFLOW` properly. Anything above a
million has to go through `recent_lsst_objects`-style keyset paging.

Measured, for the full million: ~30 s and ~98 MB without the classifier join, ~60 s and ~109 MB
with it. And `TOP n` without an `ORDER BY` is an arbitrary subset, not a sample of anything —
always order by something you mean.

In [ ]:
def top_objects(n=TAP_MAX_ROWS, order_by="o.lastmjd DESC", with_class=True,
                with_photometry=True, extra_conditions=None):
    """The top `n` LSST objects under `order_by`, in a single query (max 1,000,000).

    Useful order_by choices:
        "o.lastmjd DESC"  most recently active
        "o.n_det DESC"    best sampled
        "o.oid"           deterministic slice, pairs with keyset paging for >1M
    """
    if n > TAP_MAX_ROWS:
        raise ValueError(f"{n} exceeds the service ceiling of {TAP_MAX_ROWS}; "
                         "page with recent_lsst_objects() instead.")
    where = [f"({extra_conditions})"] if extra_conditions else []
    df = tap_query(build_query(where, order_by=order_by, limit=n,
                               with_class=with_class, with_photometry=with_photometry),
                   maxrec=n, warn=False)
    if len(df) >= n:
        print(f"Got {len(df)} rows for a limit of {n} — at or past the ceiling, "
              "so this is a truncated view of a larger result.")
    return decorate(df, with_class)


top = top_objects(1_000_000, order_by="o.lastmjd DESC")
print(len(top), "objects;", f"{top.days_ago.min():.2f}-{top.days_ago.max():.2f} days old")
top.head()

In [ ]:
top.class_name.value_counts()

#### Time-window pull, for comparison

In [ ]:
N_DAYS = 7
print(count_recent(N_DAYS), "LSST objects in the last", N_DAYS, "days")

In [ ]:
df = recent_lsst_objects(N_DAYS)
print(len(df), "objects")
df.head()

In [ ]:
df.class_name.value_counts()

In [ ]:
df.to_csv(f"alerce_tap_lsst_{N_DAYS}d.csv", index=False)

### Lightcurve for one object

The per-alert measurements are split across two tables: `detection` carries `mjd`, `band` and
position for every survey, `lsst_detection` carries the LSST photometry. Joining them server-side
is slow (~35 s for one object, versus ~1 s per table), so fetch both and merge in pandas.

In [ ]:
def lsst_lightcurve(oid):
    """Detections for one LSST object: mjd, band, PSF flux, SNR."""
    times = tap_query(f"""
        SELECT measurement_id, mjd, band, ra, dec
        FROM alerce_tap.detection
        WHERE oid = {oid} AND sid = {LSST_SID}
    """, maxrec=10_000)
    fluxes = tap_query(f"""
        SELECT measurement_id, psfflux, psffluxerr, snr, isnegative
        FROM alerce_tap.lsst_detection
        WHERE oid = {oid}
    """, maxrec=10_000)

    lc = times.merge(fluxes, on="measurement_id", how="inner")
    lc["band_name"] = lc.band.map(BANDS)
    return lc.sort_values("mjd").reset_index(drop=True)


# the best-sampled object in the frame, rather than the newest (often a single detection)
lc = lsst_lightcurve(int(df.sort_values("n_det").oid.iloc[-1]))
print(len(lc), "detections")
lc[["mjd", "band_name", "psfflux", "psffluxerr", "snr"]].head()

### Cone search

ADQL geometry works on any table with coordinates, so positional queries need no special endpoint.
`DISTANCE` returns degrees.

In [ ]:
def cone_search(ra, dec, radius_arcsec=30, maxrec=1000):
    radius_deg = radius_arcsec / 3600
    return tap_query(f"""
        SELECT oid, ra, dec, ndiasources, lastdiasourcemjdtai,
               DISTANCE(POINT('ICRS', ra, dec), POINT('ICRS', {ra}, {dec})) * 3600 AS sep_arcsec
        FROM alerce_tap.lsst_dia_object
        WHERE 1 = CONTAINS(POINT('ICRS', ra, dec),
                           CIRCLE('ICRS', {ra}, {dec}, {radius_deg}))
        ORDER BY sep_arcsec
    """, maxrec=maxrec)


cone_search(df.meanra.iloc[0], df.meandec.iloc[0], radius_arcsec=60)

### Crossmatches, and what is *not* here

`alerce_tap.xmatch` covers LSST objects (~4.1M rows), but only against two catalogues —
`catalog_id_lut` lists **AllWISE** and **Gaia DR3**, with columns `dist` and `oid_catalog`. There is
no host-galaxy association and no redshift in this schema. For LSST host galaxies with coordinates
and a redshift, Lasair's `sherlock_classifications` table remains the route (see
`queryBrokers.ipynb`).

In [ ]:
tap_query("SELECT * FROM alerce_tap.catalog_id_lut", maxrec=100)

### Other clients

```python
import pyvo
tap = pyvo.dal.TAPService("https://tap.alerce.online/tap")
job = tap.submit_job("SELECT ... ")      # async, for long queries
job.run()
```

TOPCAT: *VO → Table Access Protocol (TAP) Query*, then enter `https://tap.alerce.online/tap`.

Docs: <https://science.alerce.online/services/accessing-data-tap/>